# Doc2Vec
- Word2Vec의 확장판 모델
- 기본적인 매개변수, 속성, 메서드는 Word2Vec과 같다.
- 추가적인 부분이 생성

- 매개변수
    - dm
        - 기본값이 1
        - 학습 알고리즘을 선택
        - 1 : PV-DM
        - 0 : PT-DBOW
    - dm_mean
        - 기본값 : 0
        - PV-DM에서 문서 벡터의 계산 방식 
        - 0 : 벡터들의 합산 -> 범위가 커지는 경우가 발생 -> 노이즈로 인한 성능의 저하
        - 1 : 벡터들의 평균
    - dm_concat
        - 기본값 : 0
        - PV-DM에서 문석 벡터와 문장 벡터를 결합할 것인가?
        - 결합을 하는 경우 차원이 급격하게 증가
    - dbow-words
        - 기본값 : 0
        - PV-DBOW 사용 시 단어 벡터도 동시에 학습할 것인가? (Skip-gram 방식과 흡사)
- 속성
    - model.wv
        - 학습이 된 단어 벡터의 저장소
    - model.dv
        - 학습이 된 문장의 벡터의 저장소
    - model.dv.index_to_key
        - 문서들의 ID값 리스트
- 메서드
    - build_vocab()
        - 단어 / 문서 사진을 구성
    - train()
        - 사전이 구축이 된 뒤 직접 학습을 수행

- build_vocab(), train() 메서드를 이용하여 수동으로 학습을 설정 
- 일반적으로 Doc2Vec 객체를 생성할때 사전의 생성과 학습
- 증분 학습에서 사용이 되는 부분
    - 기존에 학습이 된 모델에 새로운 데이터셋을 추가
    - 데이터 양이 증가하면 성능이 오를수 있는 확률이 존재
    - 기존의 학습 시킨 데이터와 유사한 데이터를 증분학습하여 성능을 향상

In [5]:
# gensim 라이브러리 안에 Doc2Vec를 이용하여 임베딩
from konlpy.tag import Komoran
from gensim.models.doc2vec import Doc2Vec, TaggedDocument
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import re


In [23]:
# 샘플 데이터를 2개의 유형 문장들로 구성
docs = [
    "나는 커피를 정말 좋아한다",
    "오늘 아침에 에스프레소 두 잔을 마셨다",
    "카페라떼가 제일 맛있다고 생각한다",
    "나는 차를 더 자주 마신다",
    "녹차를 마시면 기분이 편안해진다",
    "홍차는 향이 깊고 고급스러운 느낌이다",
    "카페에서 책을 읽는 시간이 너무 좋다",
    "허브티도 몸에 좋은거 같다"
]

In [27]:
# 형태소 분석 Komoran을 이용하여 토큰화
komoran = Komoran()

# 특정 품사만 사용
allow_pos = ['NNP', 'NNG', 'VV','VA', 'MAG']
# 불용어
stop_word = ['하다', '되다', '이것', '것', '수', '거']

# 문자에서 불필요한 글자들을 제거 (정규화)
def nomalize(text):
    # 특수문자 제외, 공백에 대한 처리
    text = re.sub(r"[^가-힝0-9a-zA-Z\s\.]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

# 토큰화 함수를 정의
def tokenize(text):
    # 문자의 정규화 함수를 호출
    text = nomalize(text)
    
    tokens = []
    for word, pos in komoran.pos(text):
        if pos in allow_pos:
            # 길이를 체크하기 전에 동사, 형용사 에는 '다' 붙이기 활용
            if pos in ['VV', 'VA']:
                word += '다'
            if word not in stop_word and len(word) > 1:
                # allow_pos에 포함되어있고
                # stop_word에 포함되어있지 않으며
                # 단어의 길이가 1보다 큰 문자만 활용
                tokens.append(word)
    return tokens

In [28]:
tokenize_docs = [tokenize(doc) for doc in docs]
tokenize_docs

[['커피', '정말', '좋아하다'],
 ['오늘', '아침', '에스프레소', '마시다'],
 ['카페', '제일', '맛있다', '생각'],
 ['자주', '마시다'],
 ['녹차', '마시다', '기분', '편안'],
 ['홍차', '깊다', '고급', '느낌'],
 ['카페', '읽다', '시간', '너무', '좋다'],
 ['허브', '좋다', '같다']]

In [30]:
# 문장 별로 ID를 지정
tagged = []
for idx, toks in enumerate(tokenize_docs):
    # TaggedIocument를 이용하여 문장 당 ID를 부여하고 tagged에 추가
    tagged.append(
        TaggedDocument(words = toks, tags = [f"DOC_{idx}"])
    )
tagged

[TaggedDocument(words=['커피', '정말', '좋아하다'], tags=['DOC_0']),
 TaggedDocument(words=['오늘', '아침', '에스프레소', '마시다'], tags=['DOC_1']),
 TaggedDocument(words=['카페', '제일', '맛있다', '생각'], tags=['DOC_2']),
 TaggedDocument(words=['자주', '마시다'], tags=['DOC_3']),
 TaggedDocument(words=['녹차', '마시다', '기분', '편안'], tags=['DOC_4']),
 TaggedDocument(words=['홍차', '깊다', '고급', '느낌'], tags=['DOC_5']),
 TaggedDocument(words=['카페', '읽다', '시간', '너무', '좋다'], tags=['DOC_6']),
 TaggedDocument(words=['허브', '좋다', '같다'], tags=['DOC_7'])]